**Analyse Results**

The cell below takes `IN_DIR` pointing to the directory containing model evals as input. This directory consists of a number of csv files in the naming format *model-environment.csv* where *model* can be one of *gemini, gpt, sonnet, internvl2_5_8b, llava_onevision_7b, llama32_vision_11b* and *environment* can be one of *baseline, 4-shot, 8-shot*. Each csv file contains the headers: *file_name, label, score*. The cell below merges all csvs into a metadata.csv file outputted to `OUT_DIR` with the headings *model, environment, file_name, label, score*.

In [ ]:
import os
import pandas as pd

IN_DIR  = "./evals"
OUT_DIR = "./evals"

# Model names may themselves contain hyphens, so we match by stripping a known
# environment suffix from the right rather than splitting on the first hyphen.
VALID_MODELS = {
    "gemini", "gpt", "sonnet",
    "internvl2_5_8b", "llava_onevision_7b", "llama32_vision_11b",
}
VALID_ENVS = {"baseline", "4-shot", "8-shot"}

records = []

for fname in os.listdir(IN_DIR):
    if not fname.endswith(".csv"):
        continue

    stem = fname[:-4]

    # Identify environment by stripping its suffix; model is whatever remains.
    environment = None
    model       = None
    for candidate in VALID_ENVS:
        suffix = f"-{candidate}"
        if stem.endswith(suffix):
            environment = candidate
            model       = stem[: -len(suffix)]
            break

    if environment is None:
        print(f"Skipping unrecognised environment in: {fname}")
        continue

    if model not in VALID_MODELS:
        print(f"Skipping unrecognised model '{model}' in: {fname}")
        continue

    df = pd.read_csv(os.path.join(IN_DIR, fname))

    df.insert(0, "model",       model)
    df.insert(1, "environment", environment)

    records.append(df)

os.makedirs(OUT_DIR, exist_ok=True)

metadata = pd.concat(records, ignore_index=True)
metadata = metadata[["model", "environment", "file_name", "label", "score"]]

out_path = os.path.join(OUT_DIR, "metadata.csv")
metadata.to_csv(out_path, index=False)

print(f"Merged {len(records)} file(s) → {len(metadata)} rows")
print(f"Saved to: {out_path}")
print(metadata.head())

**Run Statistical Analyses**

The cell below takes `METADATA_PATH`, the path to a *metadata.csv* file containing the data to be analysed, as input. The csv file has the following headings: *model,environment,file_name,label,score*. The cell will run the following analyses on the data (note that the models were asked to produce a classification score between 1 and 100):

1. **Score Distributions:** Box-and-strip plots of raw confidence scores per model across environments, exposing distributional pathologies (score-ceiling effects, zero-inflation).
2. **AUC Bar Charts:** Bar chart of AUC values for all *models* across all *environments*, with a summary table.
3. **AUPRC Bar Charts:** Bar chart of AUPRC values for all *models* across all *environments*, with a summary table.
4. **Threshold Analysis:** Report precision, recall, and F1-score at the 60, 70, 80, and 90 thresholds for all *models* across all *environments*. Produce tables for this.
5. **McNemar's and DeLong's Test:** Produce per-environment model×model p-value matrices (one 6×6 matrix per environment) for each test.
6. **Expected Calibration Error (ECE):** Use confidence intervals of 10 units and compute the ECE for all *models* across all *environments*. Produce a bar chart to illustrate the results.

The cell prints all tables, matrices, and bar charts and also saves high-resolution images of these outputs to `OUT_DIR`

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score, recall_score, f1_score,
)
from scipy import stats

warnings.filterwarnings("ignore")

# ─── configuration ────────────────────────────────────────────────────────────
METADATA_PATH     = "./evals/metadata.csv"
OUT_DIR           = "./results"
THRESHOLDS        = [60, 70, 80, 90]   # applied to raw 1–100 scores
MCNEMAR_THRESHOLD = 50                 # raw score boundary for binary predictions
ECE_BIN_WIDTH     = 10                 # bin width in raw 0–100 score space

os.makedirs(OUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=1.3)

# ─── typography ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif":  ["Lora", "DejaVu Serif", "serif"],
})

FS_ANNOT        = 10   # bar value annotations
FS_TICK         = 11   # axis tick labels
FS_LEGEND       = 8    # legend item text
FS_LEGEND_TITLE = 3    # legend title ("Environment")
FS_TITLE        = 14   # subplot titles
FS_SUPTITLE     = 16   # figure suptitles
FS_STARS        = 12   # significance stars

def set_legend(ax, **kwargs):
    """Create a legend and immediately shrink its title font."""
    leg = ax.legend(**kwargs)
    leg.get_title().set_fontsize(FS_LEGEND_TITLE)
    return leg

# ─── load & normalise ─────────────────────────────────────────────────────────
df_raw = pd.read_csv(METADATA_PATH)
n_dropped = df_raw["score"].isna().sum()
df = df_raw.dropna(subset=["score"]).copy()   # .copy() prevents SettingWithCopyWarning
df["environment"] = df["environment"].replace("baseline", "0-shot")  # rename throughout
df["score_norm"]  = df["score"] / 100.0

# Canonical ground-truth labels keyed on file_name — used in get_matched so
# that the label vector is identical regardless of which model is argument a.
LABEL_REF = (
    df[["file_name", "label"]]
    .drop_duplicates("file_name")
    .set_index("file_name")["label"]
)

if n_dropped:
    print(f"⚠  Dropped {n_dropped} row(s) with null scores before analysis.")
    print(df_raw[df_raw["score"].isna()][["model", "environment", "file_name"]]
          .to_string(index=False))

# ─── model ordering & display names ───────────────────────────────────────────
MODEL_ORDER = [
    "gemini", "gpt", "sonnet",
    "llama32_vision_11b", "internvl2_5_8b", "llava_onevision_7b",
]
DISPLAY_NAMES = {
    "gemini":             "Gemini",
    "gpt":                "ChatGPT",
    "sonnet":             "Sonnet",
    "llama32_vision_11b": "Llama",
    "internvl2_5_8b":     "InternVL",
    "llava_onevision_7b": "Llava",
}

_present     = set(df["model"].unique())
MODELS       = [m for m in MODEL_ORDER if m in _present]
DISPLAY      = [DISPLAY_NAMES[m] for m in MODELS]

ENVIRONMENTS = sorted(
    df["environment"].unique(),
    key=lambda e: {"0-shot": 0, "4-shot": 1, "8-shot": 2}[e],
)
COMBOS       = [(m, e) for m in MODELS for e in ENVIRONMENTS]
N            = len(COMBOS)

MODEL_COLOURS = {
    # closed-weight
    "gemini":             "#4285F4",
    "gpt":                "#10A37F",
    "sonnet":             "#E05D44",
    # open-weight
    "internvl2_5_8b":     "#F4A261",
    "llava_onevision_7b": "#9B59B6",
    "llama32_vision_11b": "#2EC4B6",
}
ENV_LS      = {"0-shot": "-",       "4-shot": "--",     "8-shot": ":"}
ENV_COLOURS = {"0-shot": "#7fb3d3", "4-shot": "#e8a87c", "8-shot": "#82b89a"}

combo_label  = lambda m, e: f"{DISPLAY_NAMES[m]}/{e}"
COMBO_LABELS = [combo_label(m, e) for m, e in COMBOS]

# ─── data accessors ───────────────────────────────────────────────────────────
def get_subset(m, e):
    sub = df[(df["model"] == m) & (df["environment"] == e)]
    return sub["label"].values, sub["score_norm"].values

def get_matched(ca, cb):
    """Inner-join on file_name → shared (labels, scores_a, scores_b).
    Labels are taken from LABEL_REF so the vector is direction-independent.
    Both sides are deduplicated on file_name before joining.
    """
    ma, ea = ca;  mb, eb = cb
    a = (df[(df["model"] == ma) & (df["environment"] == ea)]
         [["file_name", "score_norm"]]
         .drop_duplicates("file_name")
         .rename(columns={"score_norm": "sa"}))
    b = (df[(df["model"] == mb) & (df["environment"] == eb)]
         [["file_name", "score_norm"]]
         .drop_duplicates("file_name")
         .rename(columns={"score_norm": "sb"}))
    joined = a.merge(b, on="file_name")
    joined["label"] = joined["file_name"].map(LABEL_REF)
    return joined["label"].values, joined["sa"].values, joined["sb"].values

# ─── DeLong's test (DeLong et al., 1988 — paired) ────────────────────────────
def _psi(x, y):
    """Wilcoxon kernel: 1 if x > y, 0.5 if equal, 0 otherwise."""
    return (x > y).astype(float) + 0.5 * (x == y).astype(float)

def _structural_components(labels, sa, sb):
    """Compute V10 and V01 placement-value vectors for two score arrays."""
    pos_a, neg_a = sa[labels == 1], sa[labels == 0]
    pos_b, neg_b = sb[labels == 1], sb[labels == 0]
    V10_a = np.array([np.mean(_psi(p, neg_a)) for p in pos_a])
    V01_a = np.array([np.mean(_psi(pos_a, n)) for n in neg_a])
    V10_b = np.array([np.mean(_psi(p, neg_b)) for p in pos_b])
    V01_b = np.array([np.mean(_psi(pos_b, n)) for n in neg_b])
    return V10_a, V01_a, V10_b, V01_b

def delong_p(labels, sa, sb):
    if len(np.unique(labels)) < 2:
        return np.nan
    n1, n0 = int(labels.sum()), int((1 - labels).sum())
    V10_a, V01_a, V10_b, V01_b = _structural_components(labels, sa, sb)
    Saa = np.var(V10_a, ddof=1) / n1 + np.var(V01_a, ddof=1) / n0
    Sbb = np.var(V10_b, ddof=1) / n1 + np.var(V01_b, ddof=1) / n0
    c10 = np.cov(V10_a, V10_b, ddof=1)[0, 1] if n1 > 1 else 0.0
    c01 = np.cov(V01_a, V01_b, ddof=1)[0, 1] if n0 > 1 else 0.0
    Sab = c10 / n1 + c01 / n0
    var = Saa + Sbb - 2 * Sab
    if var <= 1e-12:
        return np.nan
    z = (roc_auc_score(labels, sa) - roc_auc_score(labels, sb)) / np.sqrt(var)
    return float(2 * (1 - stats.norm.cdf(abs(z))))

# ─── McNemar's test ───────────────────────────────────────────────────────────
def mcnemar_p(labels, sa, sb, thr=MCNEMAR_THRESHOLD / 100):
    pa = (sa >= thr).astype(int)
    pb = (sb >= thr).astype(int)
    b  = int(np.sum((pa == labels) & (pb != labels)))   # A correct, B wrong
    c  = int(np.sum((pa != labels) & (pb == labels)))   # A wrong,  B correct
    if b + c == 0:
        return 1.0   # perfect agreement — no discordant pairs; p=1 by convention
    chi2 = (abs(b - c) - 1.0) ** 2 / (b + c)           # continuity-corrected
    return float(1 - stats.chi2.cdf(chi2, df=1))

# ─── ECE ──────────────────────────────────────────────────────────────────────
def compute_ece(labels, scores):
    bins = np.arange(0, 101, ECE_BIN_WIDTH)
    raw  = scores * 100
    n    = len(labels)
    ece  = 0.0
    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (raw >= lo) & (raw <= hi if hi == 100 else raw < hi)
        cnt  = mask.sum()
        acc  = float(labels[mask].mean()) if cnt else np.nan
        conf = float(scores[mask].mean()) if cnt else np.nan
        if cnt and not np.isnan(acc):
            ece += (cnt / n) * abs(acc - conf)
        rows.append({"lo": lo, "hi": hi, "count": cnt,
                     "accuracy": acc, "confidence": conf})
    return ece, pd.DataFrame(rows)

# ─── shared plot utilities ────────────────────────────────────────────────────
M     = len(MODELS)
x_pos = np.arange(M)
bw    = 0.25

def annotated_bars(ax, i_env, vals, env):
    bars = ax.bar(x_pos + (i_env - 1) * bw, vals, bw,
                  label=env, color=ENV_COLOURS[env],
                  edgecolor="white", linewidth=0.5)

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name), dpi=300, bbox_inches="tight")

# ─── pre-compute AUC, AUPRC & ECE (shared by summary table, §1, §2, and §5) ──
auc_scores   = {}
auprc_scores = {}
ece_scores   = {}

for m, e in COMBOS:
    y, s = get_subset(m, e)
    if len(np.unique(y)) < 2:
        continue
    auc_scores[(m, e)]    = roc_auc_score(y, s)
    auprc_scores[(m, e)]  = average_precision_score(y, s)
    ece_scores[(m, e)], _ = compute_ece(y, s)

# =============================================================================
# § Summary · Results Table
# =============================================================================
print("=" * 64)
print("§ Summary · Results Table")
print("=" * 64)

# MultiIndex columns: (model_display, metric) — models are the top-level axis
_col_tuples = [
    (DISPLAY_NAMES[m], metric)
    for m in MODELS
    for metric in ("AUC", "AUPRC", "ECE")
]
_cols = pd.MultiIndex.from_tuples(_col_tuples, names=["Model", "Metric"])

_data = {
    (DISPLAY_NAMES[m], metric): {
        e: round(store.get((m, e), float("nan")), 4)
        for e in ENVIRONMENTS
    }
    for m in MODELS
    for metric, store in (("AUC", auc_scores), ("AUPRC", auprc_scores), ("ECE", ece_scores))
}

summary_table = pd.DataFrame(_data, columns=_cols, index=ENVIRONMENTS)
summary_table.index.name = "Environment"

print(summary_table.to_string())
print()

# =============================================================================
# § 0 · Score Distributions
# =============================================================================
print("=" * 64)
print("§ 0 · Score Distributions")
print("=" * 64)

dist_df = df[["model", "environment", "score"]].copy()
dist_df["model_display"] = dist_df["model"].map(DISPLAY_NAMES)

fig, ax = plt.subplots(figsize=(max(10, M * 1.8), 5))
sns.boxplot(
    data=dist_df, x="model_display", y="score", hue="environment",
    order=DISPLAY, hue_order=ENVIRONMENTS,
    palette=ENV_COLOURS, linewidth=0.8, fliersize=0,
    ax=ax,
)
sns.stripplot(
    data=dist_df, x="model_display", y="score", hue="environment",
    order=DISPLAY, hue_order=ENVIRONMENTS,
    palette=ENV_COLOURS, dodge=True, size=3.5, alpha=0.55,
    jitter=True, legend=False, ax=ax,
)
ax.axhline(50, color="black", lw=0.8, ls="--", alpha=0.4, label="score = 50")
ax.set(xlabel="Model", ylabel="Confidence Score (0–100)",
       ylim=(-5, 105), title="Score Distributions by Model & Environment")
handles, labels_leg = ax.get_legend_handles_labels()
set_legend(ax, handles=handles[:len(ENVIRONMENTS)], labels=labels_leg[:len(ENVIRONMENTS)],
           title="Environment", fontsize=FS_LEGEND)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_fig(fig, "score_distributions.png")
plt.show()

# =============================================================================
# § 1–2 · Combined AUC & AUPRC Figure (two-column, full-width)
# =============================================================================
fig, (ax_auc, ax_auprc) = plt.subplots(1, 2, figsize=(max(16, M * 3.0), 5),
                                        sharey=False)

for i, e in enumerate(ENVIRONMENTS):
    annotated_bars(ax_auc,   i, [auc_scores.get((m, e),   np.nan) for m in MODELS], e)
    annotated_bars(ax_auprc, i, [auprc_scores.get((m, e), np.nan) for m in MODELS], e)

ax_auc.set(xticks=x_pos, xticklabels=DISPLAY,
           ylabel="AUC-ROC", ylim=(0, 1.1), title="AUC-ROC by Model & Environment")
ax_auprc.set(xticks=x_pos, xticklabels=DISPLAY,
             ylabel="AUPRC",  ylim=(0, 1.1), title="AUPRC by Model & Environment")

for ax in (ax_auc, ax_auprc):
    ax.tick_params(axis="x", rotation=20)

# single shared legend, placed outside to the right
handles, env_labels = ax_auc.get_legend_handles_labels()
fig.legend(handles, env_labels, title="Environment",
           fontsize=FS_LEGEND, title_fontsize=FS_LEGEND_TITLE,
           loc="center right", bbox_to_anchor=(1.0, 0.5), frameon=True)

# panel labels
for ax, label in zip((ax_auc, ax_auprc), ("(a)", "(b)")):
    ax.text(-0.08, 1.02, label, transform=ax.transAxes,
            fontsize=FS_TITLE, fontweight="bold", va="bottom")

plt.tight_layout(rect=[0, 0, 0.92, 1])   # reserve right margin for legend
save_fig(fig, "auc_auprc_combined.png")
plt.show()

# =============================================================================
# § 1 · AUC Bar Charts
# =============================================================================
print("\n" + "=" * 64)
print("§ 1 · AUC")
print("=" * 64)

# auc_scores already computed above — skipping redundant loop

for m, e in COMBOS:
    if (m, e) not in auc_scores and len(np.unique(get_subset(m, e)[0])) < 2:
        print(f"  ⚠  {DISPLAY_NAMES[m]}/{e}: single-class labels — AUC undefined, skipping.")

fig, ax = plt.subplots(figsize=(max(8, M * 1.5), 5))
for i, e in enumerate(ENVIRONMENTS):
    annotated_bars(ax, i, [auc_scores.get((m, e), np.nan) for m in MODELS], e)
ax.set(xticks=x_pos, xticklabels=DISPLAY, ylabel="AUC-ROC", ylim=(0, 1.1),
       title="AUC-ROC by Model & Environment")
set_legend(ax, title="Environment", fontsize=FS_LEGEND)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_fig(fig, "auc_bars.png")
plt.show()

auc_table = pd.DataFrame(
    {e: {DISPLAY_NAMES[m]: auc_scores.get((m, e), np.nan) for m in MODELS}
     for e in ENVIRONMENTS}
).round(4)
print("\nAUC:\n", auc_table.to_string())

# =============================================================================
# § 2 · AUPRC Bar Charts
# =============================================================================
print("\n" + "=" * 64)
print("§ 2 · AUPRC")
print("=" * 64)

# auprc_scores already computed above — skipping redundant loop

fig, ax = plt.subplots(figsize=(max(8, M * 1.5), 5))
for i, e in enumerate(ENVIRONMENTS):
    annotated_bars(ax, i, [auprc_scores.get((m, e), np.nan) for m in MODELS], e)
ax.set(xticks=x_pos, xticklabels=DISPLAY, ylabel="AUPRC", ylim=(0, 1.1),
       title="AUPRC by Model & Environment")
set_legend(ax, title="Environment", fontsize=FS_LEGEND)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_fig(fig, "auprc_bars.png")
plt.show()

auprc_table = pd.DataFrame(
    {e: {DISPLAY_NAMES[m]: auprc_scores.get((m, e), np.nan) for m in MODELS}
     for e in ENVIRONMENTS}
).round(4)
print("\nAUPRC:\n", auprc_table.to_string())

# =============================================================================
# § 3 · Threshold Analysis
# =============================================================================
print("\n" + "=" * 64)
print(f"§ 3 · Threshold Analysis  (thresholds: {THRESHOLDS})")
print("=" * 64)

thresh_rows = []
for m, e in COMBOS:
    y, s = get_subset(m, e)
    for thr in THRESHOLDS:
        pred = (s >= thr / 100).astype(int)
        thresh_rows.append({
            "model":       DISPLAY_NAMES[m],
            "environment": e,
            "threshold":   thr,
            "precision":   precision_score(y, pred, zero_division=0),
            "recall":      recall_score(y, pred, zero_division=0),
            "f1":          f1_score(y, pred, zero_division=0),
        })
thresh_df = pd.DataFrame(thresh_rows)

for thr in THRESHOLDS:
    sub = (thresh_df[thresh_df["threshold"] == thr]
           .drop(columns="threshold")
           .set_index(["model", "environment"])
           .round(4))
    print(f"\n── Threshold = {thr} " + "─" * 42)
    print(sub.to_string())

fig, axes = plt.subplots(2, 2, figsize=(max(14, M * 2.2), 10))
for ax, thr in zip(axes.flat, THRESHOLDS):
    sub = thresh_df[thresh_df["threshold"] == thr].copy()
    sub["combo"] = sub["model"] + "/" + sub["environment"]
    data = sub.set_index("combo")[["precision", "recall", "f1"]]
    sns.heatmap(data.astype(float), ax=ax, annot=True, fmt=".3f",
                cmap="YlGn", vmin=0, vmax=1,
                linewidths=0.4, cbar_kws={"shrink": 0.6})
    ax.set_title(f"Threshold = {thr}", fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=FS_TICK)
plt.suptitle("Precision / Recall / F1 by Threshold",
             fontweight="bold", fontsize=FS_SUPTITLE)
plt.tight_layout()
save_fig(fig, "threshold_analysis.png")
plt.show()

# =============================================================================
# § 4 · McNemar's Test & DeLong's Test
# =============================================================================
print("\n" + "=" * 64)
print(f"§ 4 · McNemar's Test (thr = {MCNEMAR_THRESHOLD}) & DeLong's Test")
print("=" * 64)

# ── diagnostic: expose matched-pair stats for every off-diagonal cell ─────────
print("─── get_matched diagnostics ───")
for _e in ENVIRONMENTS:
    for _i, _ma in enumerate(MODELS):
        for _j, _mb in enumerate(MODELS):
            if _i >= _j: continue
            _lbls, _sa, _sb = get_matched((_ma, _e), (_mb, _e))
            _pa   = (_sa >= MCNEMAR_THRESHOLD / 100).astype(int)
            _pb   = (_sb >= MCNEMAR_THRESHOLD / 100).astype(int)
            _b    = int(np.sum((_pa == _lbls) & (_pb != _lbls)))
            _c    = int(np.sum((_pa != _lbls) & (_pb == _lbls)))
            _cls  = np.unique(_lbls[~np.isnan(_lbls.astype(float))])
            print(f"  {DISPLAY_NAMES[_ma]:8s} vs {DISPLAY_NAMES[_mb]:8s} "
                  f"[{_e}]:  n={len(_lbls):3d}  classes={_cls}  "
                  f"b={_b}  c={_c}  b+c={_b+_c}")
print()

# ── compute one M×M matrix per environment for each test ─────────────────────
mcn_mats = {}
dl_mats  = {}

for e in ENVIRONMENTS:
    mcn_mat = np.full((M, M), np.nan)
    dl_mat  = np.full((M, M), np.nan)
    for i, ma in enumerate(MODELS):
        for j, mb in enumerate(MODELS):
            if i == j: continue
            try:
                lbls, sa, sb = get_matched((ma, e), (mb, e))
                if len(lbls) == 0:
                    continue
                mcn_mat[i, j] = mcnemar_p(lbls, sa, sb)
                if len(np.unique(lbls)) >= 2:   # roc_auc_score requires both classes
                    dl_mat[i, j] = delong_p(lbls, sa, sb)
            except Exception as ex:
                print(f"  ⚠  {DISPLAY_NAMES[ma]} vs {DISPLAY_NAMES[mb]} [{e}]: {ex}")
    mcn_mats[e] = mcn_mat
    dl_mats[e]  = dl_mat

# ── render: one figure per test, each a 1×3 row of environment sub-matrices ──
cell_w = max(7, M * 1.2)
cell_h = max(6, M * 1.0)

def merged_pval_figure(mats_by_env, suptitle, fname):
    fig, axes = plt.subplots(
        1, len(ENVIRONMENTS),
        figsize=(cell_w * len(ENVIRONMENTS), cell_h),
    )
    for ax, e in zip(axes, ENVIRONMENTS):
        mat    = mats_by_env[e]
        df_mat = pd.DataFrame(mat, index=DISPLAY, columns=DISPLAY)
        mask   = pd.DataFrame(np.isnan(mat), index=DISPLAY, columns=DISPLAY)
        sns.heatmap(df_mat, ax=ax, annot=True, fmt=".3f",
                    cmap="RdYlGn", vmin=0, vmax=1,
                    linewidths=0.4, mask=mask,
                    cbar_kws={"label": "p-value", "shrink": 0.7})
        for i_ in range(M):
            for j_ in range(M):
                p = mat[i_, j_]
                if np.isnan(p): continue
                stars = ("***" if p < 0.001 else "**" if p < 0.01
                         else "*"   if p < 0.05  else "")
                if stars:
                    ax.text(j_ + 0.5, i_ + 0.72, stars, ha="center",
                            fontsize=FS_STARS, fontweight="bold", color="black")
        ax.set_title(e, fontweight="bold", fontsize=20)
        ax.tick_params(axis="x", rotation=30, labelsize=FS_TICK)
        ax.tick_params(axis="y", rotation=0,  labelsize=FS_TICK)
    plt.tight_layout()
    fig.suptitle(suptitle, fontsize=22)
    plt.subplots_adjust(top=0.84)
    save_fig(fig, fname)
    plt.show()

merged_pval_figure(
    mcn_mats,
    f"McNemar's Test (decision threshold = {MCNEMAR_THRESHOLD})",
    "mcnemar_matrices.png",
)
merged_pval_figure(
    dl_mats,
    "DeLong's Test (AUC comparison)",
    "delong_matrices.png",
)

# =============================================================================
# § 5 · Expected Calibration Error (ECE)
# =============================================================================
print("\n" + "=" * 64)
print(f"§ 5 · ECE  (bin width = {ECE_BIN_WIDTH} score units)")
print("=" * 64)

ece_scores = ece_scores  # pre-computed above — no recomputation needed

fig, ax = plt.subplots(figsize=(max(8, M * 1.5), 5))
for i, e in enumerate(ENVIRONMENTS):
    annotated_bars(ax, i, [ece_scores.get((m, e), np.nan) for m in MODELS], e)
ax.set(xticks=x_pos, xticklabels=DISPLAY,
       ylabel="ECE",
       title="Expected Calibration Error by Model & Environment")
set_legend(ax, title="Environment", fontsize=FS_LEGEND)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_fig(fig, "ece_bars.png")
plt.show()

ece_table = pd.DataFrame(
    {e: {DISPLAY_NAMES[m]: ece_scores.get((m, e), np.nan) for m in MODELS}
     for e in ENVIRONMENTS}
).round(4)
print("\nECE (↓ = better calibrated):\n", ece_table.to_string())

print(f"\n✓  All figures written to '{OUT_DIR}'")